# Privileged debrief (mode 4)

Interactively analyze any **recorded** play conversation with full access to
ground truth: the saved snapshot images *and* the exact Settings JSON
(coordinates, angles) that the playing agent never saw.

Things to know:

- **The playing agent never sees this conversation.** Debriefs are stored as
  their own NAMS conversations (`kind='debrief'`, linked to the play
  conversation via a `DEBRIEF_OF` edge). The only way debrief content reaches
  the play conversation is the explicit **Save self-eval** button, which
  distills the debrief into a mode-3-style verdict
  (`kind='self_evaluation'` message + reasoning trace).
- **Switching the play conversation starts a fresh debrief conversation.**
  The previous one stays stored in NAMS; this panel just stops appending to it.
- **The model can call `[SHOW <step>]`** to pull up any recorded step's
  before/after frames + settings, so multi-generation replies are normal —
  you'll see each generation and the fetched frames as they happen.
- **GPU caveat:** this notebook loads its own Gemma copy (~16 GB). If the play
  notebook is open with its model loaded, close it (or run on a box with VRAM
  for two).

Prereqs: Neo4j is up, and there is at least one recorded play conversation.

In [ ]:
import os
import sys

# Run from the repo root so `agent` imports and `.env` resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from agent.debrief import DebriefSession

# Connects NAMS on a background loop and loads Gemma 4 E4B (first run is slow).
# Logging is on by default: every LLM call and DB retrieval lands under
# session.logger.run_dir.
session = DebriefSession()
if session.logger is not None:
    print("log dir:", session.logger.run_dir)

In [ ]:
import ipywidgets as widgets
from IPython.display import Image, display

FRAME_WIDTH = 420  # px; frames are 224x224 native, upscaled for visibility

picker_out = widgets.Output()


def _conversation_options():
    """(label, session_id) pairs for the dropdown -- play conversations only."""
    options = []
    for c in session.list_conversations():
        sid = c["session_id"]
        label = f"{sid[:8]}… — {c.get('created_at')}, {c.get('n_messages')} msgs"
        options.append((label, sid))
    return options


dropdown = widgets.Dropdown(
    options=_conversation_options(),
    value=None,
    description="Play conv:",
    layout=widgets.Layout(width="600px"),
)
refresh_btn = widgets.Button(description="Refresh list", button_style="info")


def _on_select(change):
    if change["new"] is None:
        return
    with picker_out:
        picker_out.clear_output()
        info = session.select(change["new"])
        print(f"Analyzing play conversation: {info['play_session_id']}")
        print(f"New debrief conversation:    {info['debrief_session_id']}")
        print(
            f"({info['n_messages']} messages, {info['n_moves']} moves, "
            f"{info['n_snapshots']} snapshots)"
        )
        if info["latest_frame_path"]:
            print("-- latest saved frame (current state) --")
            display(Image(filename=info["latest_frame_path"], width=FRAME_WIDTH))


def _on_refresh(_):
    # Sessions recorded after notebook start appear here; the current
    # selection (and its debrief conversation) is untouched.
    dropdown.unobserve(_on_select, names="value")
    selected = dropdown.value
    dropdown.options = _conversation_options()
    dropdown.value = selected if selected in [v for _, v in dropdown.options] else None
    dropdown.observe(_on_select, names="value")


dropdown.observe(_on_select, names="value")
refresh_btn.on_click(_on_refresh)

display(widgets.VBox([widgets.HBox([dropdown, refresh_btn]), picker_out]))

In [ ]:
question_box = widgets.Textarea(
    value="",
    placeholder="Ask about the recorded session (e.g. 'Was the agent ever facing the gold?')...",
    description="You:",
    layout=widgets.Layout(width="600px", height="70px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
save_btn = widgets.Button(description="Save self-eval", button_style="warning")
chat_out = widgets.Output()


def _show_generation(step):
    """Render one generation of a debrief turn as it happens (wired to
    ``session.ask(on_step=...)``)."""
    print(f"Analyst: {step['text']}")
    if step["show_step"] is not None:
        print(f"[tool call: SHOW {step['show_step']}]")
        for path in step["frames"]:
            display(Image(filename=path, width=FRAME_WIDTH))
        if not step["frames"]:
            print("(no such step -- the model was told the valid range)")


def _on_ask(_):
    q = question_box.value.strip()
    if not q:
        return
    ask_btn.disabled = True
    try:
        with chat_out:
            print(f"\n=== You: {q}")
            result = session.ask(q, on_step=_show_generation)
            print(
                f"[turn ended: {result['num_generations']} generation(s), "
                f"{result['tool_calls']} SHOW call(s)]"
            )
        question_box.value = ""
    finally:
        ask_btn.disabled = False


def _on_save(_):
    save_btn.disabled = True
    try:
        with chat_out:
            info = session.save_self_eval()
            print("\n*** Self-eval verdict saved to the play conversation ***")
            print(f"message id: {info['eval_message_id']}")
            print(f"trace id:   {info['trace_id']}")
            print(f"--- verdict ---\n{info['verdict']}")
    finally:
        save_btn.disabled = False


ask_btn.on_click(_on_ask)
save_btn.on_click(_on_save)

display(widgets.VBox([question_box, widgets.HBox([ask_btn, save_btn]), chat_out]))

When you're done, release the model and close the memory client:

In [ ]:
session.close()